# Curva ROC y ROC-AUC: detección de piezas defectuosas

<a href="https://colab.research.google.com/" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

En los cuadernos anteriores evaluamos clases predichas mediante la matriz de confusión, Precision, Recall y F1-score. Ahora trabajaremos con las **probabilidades** del modelo para estudiar muchos umbrales mediante la **curva ROC**. Usaremos un caso sintético de inspección industrial: detectar piezas defectuosas.


## Objetivos del cuaderno

Al finalizar podrás:

- distinguir una clase predicha de una probabilidad estimada;
- explicar cómo un umbral transforma probabilidades en decisiones;
- calcular sensibilidad, especificidad y tasa de falsos positivos;
- interpretar los ejes y puntos de una curva ROC;
- calcular e interpretar ROC-AUC;
- entender por qué ROC-AUC no elige por sí solo el umbral operativo;
- reconocer el efecto del desbalance de clases y la utilidad de Precision-Recall.


## 1. Importar las herramientas

Este cuaderno no requiere archivos externos ni instalaciones adicionales en Google Colab.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


**Interpretación de la salida:** esta celda carga las herramientas para todo el cuaderno. Además de generar datos y entrenar modelos, importamos funciones que calculan FPR, TPR, ROC-AUC y la curva Precision-Recall. La semilla fija hace que las particiones y los resultados sean reproducibles al volver a ejecutar el notebook.


## 2. Preparar un problema de clasificación

Generaremos mediciones simuladas de piezas. La clase positiva (1) significa **defectuosa** y la clase negativa (0), **correcta**. La proporción de defectos será menor que la de piezas correctas para representar una situación industrial razonable.


In [ ]:
X, y = make_classification(
    n_samples=1400, n_features=8, n_informative=5, n_redundant=1,
    n_clusters_per_class=2, weights=[0.86, 0.14],
    class_sep=0.9, flip_y=0.02, random_state=RANDOM_STATE,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE,
)
print(f'Piezas totales: {len(y):,}')
print(f'Defectuosas: {y.sum():,} ({y.mean():.1%})')
print(f'Defectuosas en prueba: {y_test.sum():,} de {len(y_test):,}')


**Interpretación de la salida:** el tamaño total indica cuántas piezas simuladas se analizarán. La proporción de defectuosas muestra que la clase positiva es minoritaria, por lo que Accuracy podría resultar engañosa si se interpreta sin revisar Recall, Precision y la matriz de confusión. El conteo de defectos en prueba es la cantidad disponible para evaluar la capacidad de detección fuera del entrenamiento.


La división estratificada conserva una proporción semejante de defectos en entrenamiento y prueba. La semilla permite repetir el ejemplo con los mismos resultados.


In [ ]:
modelo_logistico = Pipeline([
    ('escalador', StandardScaler()),
    ('logistica', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
modelo_logistico.fit(X_train, y_train)
print('Modelo entrenado.')


**Interpretación de la salida:** “Modelo entrenado” confirma que el Pipeline pudo ajustar la estandarización y la regresión logística. A partir de este punto, el modelo ya puede producir una puntuación de defecto para cada pieza; todavía no hemos decidido qué puntuación será suficiente para generar una alerta.


La regresión logística será una fuente de probabilidades. El objetivo no es optimizar el algoritmo, sino estudiar cómo sus puntuaciones se convierten en decisiones y puntos de la curva ROC.


## 3. Comparar clases y probabilidades

`predict()` devuelve una clase final. `predict_proba()` devuelve una probabilidad para cada clase. La curva ROC necesita las probabilidades o puntuaciones, porque recorre muchos umbrales.


In [ ]:
probabilidades = modelo_logistico.predict_proba(X_test)
clases_modelo = modelo_logistico.named_steps['logistica'].classes_
print('Orden de las clases:', clases_modelo)
print('Forma de la matriz de probabilidades:', probabilidades.shape)


**Interpretación de la salida:** la primera dimensión de la matriz corresponde a piezas y la segunda a clases. El orden de clases confirma qué columna debemos usar como puntuación positiva: la columna asociada con la clase 1, “defectuosa”. Elegir la columna equivocada invertiría la interpretación de la ROC y del AUC.


La columna 0 corresponde a la clase correcta y la columna 1 a la clase defectuosa. Usaremos la segunda columna como puntuación positiva.


In [ ]:
prob_defecto = probabilidades[:, 1]
pred_umbral_05 = (prob_defecto >= 0.5).astype(int)

tabla_probabilidades = pd.DataFrame({
    'clase_real': np.where(y_test == 1, 'Defectuosa', 'Correcta'),
    'P_correcta': probabilidades[:, 0],
    'P_defecto': prob_defecto,
    'prediccion_umbral_05': np.where(pred_umbral_05 == 1, 'Defectuosa', 'Correcta'),
})
tabla_probabilidades.sort_values('P_defecto').iloc[[0, 1, -2, -1]].round(3)


**Interpretación de la salida:** la tabla permite comparar la clase real con las dos probabilidades y la decisión producida por el umbral 0.5. Una pieza con probabilidad cercana a 0.5 es ambigua; una cercana a 0 o 1 está más separada del punto de corte. Dos piezas pueden compartir la misma etiqueta final y, sin embargo, tener distintos niveles de riesgo.


Dos piezas pueden recibir la misma clase final y tener probabilidades muy diferentes. Esa información adicional se conserva mientras trabajamos con probabilidades, pero se pierde al convertirlas inmediatamente en 0 o 1.


In [ ]:
sumas_por_fila = probabilidades.sum(axis=1)
predicciones_modelo = modelo_logistico.predict(X_test)
print('Suma mínima de probabilidades:', f'{sumas_por_fila.min():.3f}')
print('Suma máxima de probabilidades:', f'{sumas_por_fila.max():.3f}')
print('¿predict coincide con umbral 0.5?', np.array_equal(predicciones_modelo, pred_umbral_05))


**Interpretación de la salida:** las probabilidades de las dos clases suman 1 en cada fila, como debe ocurrir en una clasificación binaria. La coincidencia entre `predict()` y el umbral 0.5 valida la regla de decisión predeterminada. La ROC irá más allá de esa única decisión y evaluará todo el rango de umbrales.


En un problema binario las probabilidades suman 1. Además, `predict()` coincide con aplicar el umbral 0.5, pero la ROC analizará muchos otros umbrales.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(prob_defecto[y_test == 0], bins=np.linspace(0, 1, 16), alpha=0.65, label='Correcta', color='#4C78A8')
plt.hist(prob_defecto[y_test == 1], bins=np.linspace(0, 1, 16), alpha=0.65, label='Defectuosa', color='#E45756')
plt.axvline(0.5, color='black', linestyle='--', label='Umbral 0.5')
plt.title('Distribución de probabilidades por clase real')
plt.xlabel('Probabilidad estimada de defecto')
plt.ylabel('Número de piezas')
plt.legend()
plt.show()


**Interpretación de la salida:** el histograma muestra cuánto se solapan las puntuaciones de piezas correctas y defectuosas. La zona de solapamiento representa casos difíciles: ningún umbral separará perfectamente ambas clases. Mover la línea de 0.5 cambia cuántas piezas se envían a revisión y cuántos defectos podrían pasar desapercibidos.


## 4. Mover el umbral de clasificación

Cambiar el umbral no vuelve a entrenar el modelo: cambia la regla de decisión. Para cada umbral calcularemos VP, FP, FN, VN, sensibilidad, especificidad y tasa de falsos positivos.


In [ ]:
umbrales = [0.10, 0.25, 0.40, 0.50, 0.70, 0.90]
resultados_umbrales = []
for umbral in umbrales:
    predicciones = (prob_defecto >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_test, predicciones, labels=[0, 1]).ravel()
    sensibilidad = vp / (vp + fn)
    especificidad = vn / (vn + fp)
    resultados_umbrales.append({
        'umbral': umbral, 'VN': vn, 'FP': fp, 'FN': fn, 'VP': vp,
        'sensibilidad': sensibilidad, 'especificidad': especificidad,
        'tasa_FP': 1 - especificidad,
        'precision': precision_score(y_test, predicciones, zero_division=0),
        'F1': f1_score(y_test, predicciones, zero_division=0),
    })
tabla_umbrales = pd.DataFrame(resultados_umbrales)
tabla_umbrales.round(3)


**Interpretación de la salida:** cada fila representa una política de inspección distinta. VN y VP son aciertos; FP son piezas correctas marcadas para revisión; FN son defectos que el sistema no detectó. La sensibilidad/TPR mide la cobertura de defectos, la especificidad mide la cobertura de piezas correctas y la FPR indica qué fracción de piezas correctas se convierte en falsa alarma.


Al bajar el umbral, el modelo suele declarar más piezas como defectuosas: sube la sensibilidad, pero también puede subir la tasa de falsos positivos. Al subirlo, las alertas son más exigentes y pueden quedar defectos sin detectar.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tabla_umbrales['umbral'], tabla_umbrales['sensibilidad'], marker='o', label='Sensibilidad / TPR')
plt.plot(tabla_umbrales['umbral'], tabla_umbrales['especificidad'], marker='o', label='Especificidad')
plt.plot(tabla_umbrales['umbral'], tabla_umbrales['tasa_FP'], marker='o', label='Tasa FP / FPR')
plt.title('Sensibilidad, especificidad y FPR según el umbral')
plt.xlabel('Umbral para declarar defecto')
plt.ylabel('Valor')
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.show()


**Interpretación de la salida:** las líneas permiten observar el intercambio entre detectar defectos y evitar falsas alarmas. En general, al disminuir el umbral aumenta la sensibilidad porque se aceptan más alertas, pero también puede aumentar la FPR. El punto operativo debe elegirse considerando la capacidad de revisión y el costo de liberar un defecto.


## 5. Calcular sensibilidad y especificidad

Retomemos el umbral 0.5 para reconstruir las tasas desde la matriz de confusión.


<div style="background-color:#d9edf7;border-left:6px solid #31708f;padding:12px;font-size:1.1em"><b>Fórmulas clave</b><br><br>$$Sensibilidad = TPR = \frac{VP}{VP + FN}$$<br><br>$$Especificidad = \frac{VN}{VN + FP}$$<br><br>$$FPR = 1 - Especificidad = \frac{FP}{FP + VN}$$</div>


In [ ]:
vn_05, fp_05, fn_05, vp_05 = confusion_matrix(y_test, pred_umbral_05, labels=[0, 1]).ravel()
sensibilidad_manual = vp_05 / (vp_05 + fn_05)
especificidad_manual = vn_05 / (vn_05 + fp_05)
tasa_fp_manual = fp_05 / (fp_05 + vn_05)

pd.DataFrame({
    'VN': [vn_05], 'FP': [fp_05], 'FN': [fn_05], 'VP': [vp_05],
    'sensibilidad_TPR': [sensibilidad_manual],
    'especificidad': [especificidad_manual],
    'tasa_FP_FPR': [tasa_fp_manual],
}).round(3)


**Interpretación de la salida:** la tabla reconstruye las cantidades de la matriz de confusión con el umbral 0.5 y aplica las fórmulas de TPR, especificidad y FPR. La relación más importante es FPR = 1 - especificidad: una especificidad alta significa que pocas piezas correctas generan alertas falsas.


La sensibilidad es igual al Recall: mide qué proporción de defectos reales detectamos. La especificidad mide qué proporción de piezas correctas dejamos pasar sin generar alerta. La FPR es el complemento de la especificidad.


In [ ]:
print('Sensibilidad / Recall:', f'{sensibilidad_manual:.3f}', f'(Recall: {recall_score(y_test, pred_umbral_05):.3f})')
print('Tasa de falsos positivos:', f'{tasa_fp_manual:.3f}')
print('1 - especificidad:', f'{1 - especificidad_manual:.3f}')


**Interpretación de la salida:** la sensibilidad obtenida aquí debe coincidir con Recall, porque ambas expresiones calculan VP/(VP+FN). La FPR y el complemento de la especificidad también deben coincidir. Esta comprobación conecta los nombres usados en clasificación con los ejes que aparecerán en la Curva ROC.


## 6. Construir la Curva ROC

Cada umbral produce un punto formado por:

- eje horizontal: **FPR**, tasa de falsos positivos;
- eje vertical: **TPR**, sensibilidad o Recall.

La curva ROC reúne esos puntos para muchos umbrales. Un modelo útil se acerca a la esquina superior izquierda: alta sensibilidad y baja FPR.


In [ ]:
puntos_seleccionados = tabla_umbrales[['umbral', 'tasa_FP', 'sensibilidad', 'FP', 'FN']].copy()
puntos_seleccionados.round(3)


**Interpretación de la salida:** cada fila es un punto de la ROC generado por un umbral concreto. La columna tasa_FP será la coordenada horizontal y sensibilidad será la coordenada vertical. Un punto más arriba y más a la izquierda suele ser preferible, pero la decisión final depende de cuánto cueste cada FP y FN.


La tabla muestra algunos puntos de la curva. No existe un único punto correcto de forma universal: cada punto representa una política diferente para decidir qué piezas enviar a revisión.


In [ ]:
fpr_log, tpr_log, umbrales_roc = roc_curve(y_test, prob_defecto)

plt.figure(figsize=(7, 6))
plt.plot(fpr_log, tpr_log, linewidth=2, label='Regresión logística')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Referencia al azar')
plt.scatter([tasa_fp_manual], [sensibilidad_manual], color='black', s=70, zorder=3, label='Umbral 0.5')
plt.title('Curva ROC: detección de piezas defectuosas')
plt.xlabel('Tasa de falsos positivos (FPR = 1 - especificidad)')
plt.ylabel('Sensibilidad (TPR = Recall)')
plt.xlim(0, 1); plt.ylim(0, 1.02)
plt.grid(alpha=0.25); plt.legend(loc='lower right')
plt.show()


**Interpretación de la salida:** la curva muestra el desempeño del modelo en muchos umbrales, no solo en 0.5. La diagonal representa comportamiento aleatorio. El punto negro localiza el umbral 0.5; otros puntos de la línea corresponden a políticas de decisión diferentes. Una curva cercana a la esquina superior izquierda indica mejor separación entre defectuosas y correctas.


La diagonal representa un clasificador parecido al azar. Cuanto más cerca esté la curva de la esquina superior izquierda, mejor separa el modelo las piezas defectuosas de las correctas en distintos umbrales.


## 7. Resumir la curva mediante ROC-AUC

ROC-AUC es el área bajo la curva ROC. Se calcula con probabilidades o puntuaciones, no con una única columna de clases predichas.


<div style="background-color:#fff3cd;border-left:6px solid #f0ad4e;padding:12px;font-size:1.1em"><b>Interpretación clave</b><br><br>$$ROC	ext{-}AUC = \int_0^1 TPR(FPR)\,dFPR$$<br><br>También puede interpretarse como la probabilidad de que un positivo real reciba una puntuación mayor que un negativo real elegido al azar.</div>


In [ ]:
auc_logistico = roc_auc_score(y_test, prob_defecto)
print('ROC-AUC de la regresión logística:', f'{auc_logistico:.3f}')


**Interpretación de la salida:** el ROC-AUC condensa el área completa bajo la curva en un solo valor. Un valor cercano a 1 significa que el modelo suele ordenar las piezas defectuosas por encima de las correctas; 0.5 equivale aproximadamente al azar. Este valor evalúa separación global, no el costo ni el volumen de alertas de un umbral específico.


Un ROC-AUC cercano a 1 indica una buena capacidad general de separación; un valor cercano a 0.5 es compatible con el azar. AUC resume muchos umbrales, pero no indica por sí solo cuántas alertas habrá con un umbral concreto.


In [ ]:
prob_positivos = prob_defecto[y_test == 1]
prob_negativos = prob_defecto[y_test == 0]
conteo_pares = 0
for positiva in prob_positivos:
    for negativa in prob_negativos:
        if positiva > negativa:
            conteo_pares += 1
        elif positiva == negativa:
            conteo_pares += 0.5
auc_por_pares = conteo_pares / (len(prob_positivos) * len(prob_negativos))
print('AUC por comparación de pares:', f'{auc_por_pares:.3f}')


**Interpretación de la salida:** este cálculo alternativo compara pares de piezas: una defectuosa y una correcta. La proporción de pares correctamente ordenados debe aproximarse al ROC-AUC. Esta lectura ayuda a entender que el AUC evalúa el ranking de puntuaciones, no la calibración exacta de cada probabilidad.


La comparación de pares comprueba la interpretación probabilística del AUC: se cuenta con qué frecuencia una pieza defectuosa recibe una puntuación mayor que una pieza correcta.


## 8. Comparar dos modelos conocidos

Entrenaremos un árbol pequeño sobre los mismos datos. No buscamos optimizarlo, sino comprobar que ROC y AUC permiten comparar la capacidad de separación de dos modelos.


In [ ]:
modelo_arbol = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
modelo_arbol.fit(X_train, y_train)
prob_arbol = modelo_arbol.predict_proba(X_test)[:, 1]
auc_arbol = roc_auc_score(y_test, prob_arbol)

pd.DataFrame({
    'modelo': ['Regresión logística', 'Árbol de decisión'],
    'ROC_AUC': [auc_logistico, auc_arbol],
}).round(3)


**Interpretación de la salida:** la tabla compara el ROC-AUC de ambos modelos usando exactamente el mismo conjunto de prueba. El modelo con mayor AUC ordena mejor, en promedio, los positivos por encima de los negativos. Aun así, el AUC no garantiza que ese modelo sea mejor en el umbral operativo que la planta necesita.


**Explicación del bloque:** este bloque continúa el paso anterior y muestra el resultado calculado o visualizado para poder interpretarlo antes de avanzar.


In [ ]:
fpr_arbol, tpr_arbol, _ = roc_curve(y_test, prob_arbol)
plt.figure(figsize=(7, 6))
plt.plot(fpr_log, tpr_log, linewidth=2, label=f'Logística (AUC={auc_logistico:.3f})')
plt.plot(fpr_arbol, tpr_arbol, linewidth=2, label=f'Árbol (AUC={auc_arbol:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Azar')
plt.title('Comparación de curvas ROC')
plt.xlabel('FPR'); plt.ylabel('TPR / Sensibilidad')
plt.xlim(0, 1); plt.ylim(0, 1.02)
plt.grid(alpha=0.25); plt.legend(loc='lower right'); plt.show()


**Interpretación de la salida:** las dos curvas permiten ver no solo quién tiene mayor AUC, sino dónde cada modelo puede ofrecer una combinación útil de TPR y FPR. Los escalones del árbol reflejan que entrega pocas probabilidades distintas; la logística suele producir una gradación más continua.


Un AUC mayor indica mejor ordenamiento general en esta separación de entrenamiento y prueba. No es una conclusión universal: los resultados dependen de los datos, las variables y la configuración.


## 9. ROC-AUC no elige el umbral

El AUC resume el ordenamiento general. La matriz de confusión describe una decisión concreta. El AUC no cambia al modificar el umbral, pero sí cambian VP, FP, FN y VN.


In [ ]:
resumen_decisiones = tabla_umbrales[tabla_umbrales['umbral'].isin([0.25, 0.50, 0.90])].copy()
resumen_decisiones['ROC_AUC_modelo'] = auc_logistico
resumen_decisiones[['umbral', 'VP', 'FP', 'FN', 'sensibilidad', 'tasa_FP', 'precision', 'F1', 'ROC_AUC_modelo']].round(3)


**Interpretación de la salida:** el ROC-AUC aparece repetido para los tres umbrales porque el modelo y sus probabilidades no cambiaron. Lo que sí cambia son VP, FP, FN, sensibilidad, FPR, Precision y F1. Por eso el AUC sirve para evaluar la separación general, pero no sustituye la selección de un umbral con consecuencias operativas explícitas.


El mismo modelo y sus mismas probabilidades pueden operar con políticas distintas. En producción, el umbral debe relacionarse con costos, capacidad de revisión, seguridad y consecuencias de liberar un defecto.


### Limitaciones de ROC-AUC

| ROC-AUC aporta | ROC-AUC no resuelve por sí solo |
|---|---|
| Capacidad general de separar clases | Qué umbral debe usarse |
| Comparación del ordenamiento entre modelos | Cuántos FP y FN habrá en una decisión concreta |
| Evaluación a través de muchos umbrales | El costo operativo de cada error |


## 10. Una mirada a clases muy desbalanceadas

Crearemos otro conjunto donde solo cerca del 2% de las piezas sean defectuosas. Sirve para observar por qué conviene complementar ROC-AUC con Precision, Recall y la curva Precision-Recall.


In [ ]:
X_raro, y_raro = make_classification(
    n_samples=5000, n_features=10, n_informative=5, n_redundant=2,
    weights=[0.98, 0.02], class_sep=1.0, flip_y=0.01,
    random_state=RANDOM_STATE,
)
X_raro_train, X_raro_test, y_raro_train, y_raro_test = train_test_split(
    X_raro, y_raro, test_size=0.30, stratify=y_raro, random_state=RANDOM_STATE,
)
modelo_raro = Pipeline([
    ('escalador', StandardScaler()),
    ('logistica', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
modelo_raro.fit(X_raro_train, y_raro_train)
print('Positivos en prueba:', int(y_raro_test.sum()), 'de', len(y_raro_test))


**Interpretación de la salida:** este bloque crea un escenario de alta prevalencia de piezas correctas y muy pocos defectos. El número absoluto de positivos en prueba es importante: con pocos defectos, incluso un modelo razonable puede generar muchas alertas falsas si la FPR no es suficientemente baja.


**Explicación del bloque:** este bloque continúa el paso anterior y muestra el resultado calculado o visualizado para poder interpretarlo antes de avanzar.


In [ ]:
prob_raro = modelo_raro.predict_proba(X_raro_test)[:, 1]
auc_raro = roc_auc_score(y_raro_test, prob_raro)
ap_raro = average_precision_score(y_raro_test, prob_raro)
pred_raro = (prob_raro >= 0.10).astype(int)

print('ROC-AUC:', f'{auc_raro:.3f}')
print('Average Precision:', f'{ap_raro:.3f}')
print('Precision con umbral 0.10:', f'{precision_score(y_raro_test, pred_raro, zero_division=0):.3f}')
print('Recall con umbral 0.10:', f'{recall_score(y_raro_test, pred_raro, zero_division=0):.3f}')


**Interpretación de la salida:** aquí se comparan ROC-AUC y Average Precision con Precision y Recall en un umbral concreto. Un ROC-AUC aceptable puede coexistir con Precision baja cuando los positivos son raros: muchas piezas correctas pueden ocupar el conjunto de falsas alarmas. Average Precision resume la perspectiva Precision-Recall y ayuda a evaluar la confiabilidad de las alertas.


Con pocos positivos, una FPR pequeña puede representar muchas falsas alarmas en cantidad absoluta. Por eso ROC-AUC debe acompañarse de Precision, Recall y, cuando la clase positiva es rara, la curva Precision-Recall.


In [ ]:
fpr_raro, tpr_raro, _ = roc_curve(y_raro_test, prob_raro)
precision_rara, recall_raro, _ = precision_recall_curve(y_raro_test, prob_raro)

fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
ejes[0].plot(fpr_raro, tpr_raro, label=f'ROC-AUC={auc_raro:.3f}')
ejes[0].plot([0, 1], [0, 1], '--', color='gray')
ejes[0].set_title('ROC con clase positiva rara')
ejes[0].set_xlabel('FPR'); ejes[0].set_ylabel('TPR / Recall')
ejes[0].legend(); ejes[0].grid(alpha=0.25)
ejes[1].plot(recall_raro, precision_rara, color='#E45756', label=f'AP={ap_raro:.3f}')
ejes[1].set_title('Precision-Recall con clase positiva rara')
ejes[1].set_xlabel('Recall'); ejes[1].set_ylabel('Precision')
ejes[1].legend(); ejes[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()


**Interpretación de la salida:** la figura izquierda muestra la separación global mediante FPR y TPR; la derecha muestra cómo cambia la confiabilidad de las alertas al buscar mayor cobertura. En el caso desbalanceado, Precision-Recall puede revelar problemas que no son tan visibles en la ROC, por lo que conviene revisar ambas curvas junto con la matriz de confusión.


ROC observa TPR frente a FPR. Precision-Recall se concentra en la clase positiva y en la confiabilidad de las alertas. En un proceso donde los defectos son muy raros, esta segunda perspectiva puede ser especialmente informativa.


## 11. Una pequeña exploración

Modifica la lista de umbrales y observa cómo se desplazan los puntos del modelo principal. No busques automáticamente el “mejor” valor: relaciona cada punto con las consecuencias de sus errores.


In [ ]:
umbrales_actividad = [0.15, 0.35, 0.55, 0.75, 0.95]
actividad = []
for umbral in umbrales_actividad:
    predicciones = (prob_defecto >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_test, predicciones, labels=[0, 1]).ravel()
    especificidad = vn / (vn + fp)
    actividad.append({
        'umbral': umbral, 'FPR': 1 - especificidad,
        'TPR_sensibilidad': vp / (vp + fn),
        'precision': precision_score(y_test, predicciones, zero_division=0),
        'F1': f1_score(y_test, predicciones, zero_division=0),
    })
pd.DataFrame(actividad).round(3)


**Interpretación de la salida:** cada fila corresponde a uno de los umbrales elegidos para la actividad. FPR y TPR ubican el punto en el plano ROC, mientras Precision y F1 ayudan a juzgar la confiabilidad de las alertas y el equilibrio. Compara las filas con los costos reales antes de recomendar un umbral.


Preguntas para explorar:

- ¿qué umbral reduce más la FPR?
- ¿qué punto ofrece mayor TPR?
- ¿qué punto queda más cerca de la esquina superior izquierda?
- ¿ese criterio geométrico coincide con las necesidades de la planta?
- ¿qué matriz de confusión produce cada punto?


## Cierre

Las probabilidades permiten estudiar un clasificador más allá de una única clase final. El umbral transforma esas probabilidades en decisiones y modifica el equilibrio entre sensibilidad y especificidad. La curva ROC reúne ese comportamiento para muchos umbrales, mientras que ROC-AUC resume la capacidad general de separación.

La curva ROC es una herramienta de evaluación; la elección del umbral requiere además contexto, costos, validación con datos nuevos y revisión de errores.


## Para pensar

1. ¿Qué información conserva una probabilidad que se pierde al convertirla inmediatamente en una clase?
2. ¿Por qué bajar el umbral suele aumentar la sensibilidad y reducir la especificidad?
3. ¿Qué representa cada punto de una curva ROC?
4. ¿Qué significa que un modelo tenga ROC-AUC cercano a 0.5?
5. ¿Por qué un ROC-AUC alto no garantiza Precision alta cuando la clase positiva es muy rara?
